# Module 1 — OCR Văn Bản Tiếng Việt (PDF Scan / Ảnh)

Notebook này thực hiện nhận dạng văn bản tiếng Việt từ ảnh scan.
- Bảo tồn đầy đủ dấu thanh tiếng Việt Unicode.
- Khôi phục checkpoint khi ngắt quãng.
- Xuất ra `viet.txt` UTF-8 và sidecar JSON metadata.

In [ ]:
# 1. Cài đặt và nạp thư viện
import os
import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve()
if (REPO_ROOT / 'OCR').exists():
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
elif (REPO_ROOT.parent / 'OCR').exists():
    if str(REPO_ROOT.parent) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT.parent))

from PIL import Image
from OCR.utils.ocr_pipeline import BaseOCREngine, OCRPipeline
from OCR.utils.checkpoints import OCRCheckpointManager
from OCR.utils.export_text import export_ocr_result
print("Đã nạp OCR pipeline thành công!")

In [ ]:
# 2. Cấu hình nguồn tài liệu và chế độ OCR
INPUT_IMAGE_OR_PDF = "tests/fixtures/sample_viet_scan.png"
MODEL_NAME = "PaddleOCR-Viet"
USE_GPU = False
OUTPUT_DIR = Path("data/ocr_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Khởi tạo Engine và Chạy Nhận Dạng
engine = BaseOCREngine(model_name="viet", use_gpu=USE_GPU)
pipeline = OCRPipeline(engine=engine, reading_order="ltr_horizontal")

if INPUT_IMAGE_OR_PDF.endswith(".pdf"):
    from OCR.utils.render_pdf import render_pdf_to_images
    images = render_pdf_to_images(INPUT_IMAGE_OR_PDF)
else:
    images = [Image.open(INPUT_IMAGE_OR_PDF)]

print(f"Bắt đầu nhận dạng {len(images)} trang...")
result = pipeline.process_images(images, source_id="viet_scan_doc")

print(f"=== KẾT QUẢ OCR TIẾNG VIỆT ===")
print(f"Engine: {result['engine_info']['engine_type']}")
print(f"Phần cứng: {result['engine_info']['hardware']}")
print(f"Số ký tự nhận dạng: {len(result['full_text'])}")
print("\n--- VĂN BẢN TRÍCH XUẤT ---")
print(result['full_text'])

In [ ]:
# 4. Xuất file viet.txt và metadata
out_txt = OUTPUT_DIR / "viet.txt"
out_meta = OUTPUT_DIR / "viet_metadata.json"
export_ocr_result(result, out_txt, out_meta)
print(f"Đã xuất kết quả: {out_txt}")
print(f"Đã xuất metadata: {out_meta}")